# Task 4: Value of Adaptive Pricing (Section 3.3)

This notebook implements **Task 4 — Value of Adaptive Pricing** as described in Section 3.3 of the paper.

## Setup

We compare two recourse policies under supply disruption:

- **Scenario a — Full recourse (adaptive pricing):** After a disruption ε is realised, the provider jointly optimises *both* the allocation y *and* the prices p. Prices are recovered at optimality via eq. (4):

  $$p_i = v_i \left(1 - \sum_{j} y_{ij}\right)$$

  This is the inner SOCP of Corollary 1 (eq. 15).  The provider has full flexibility to adapt prices to the disrupted capacity.

- **Scenario b — Fixed-price recourse (allocation only):** Prices are *locked* at pre-disruption nominal values $\hat{p}_i$.  Under disruption the provider can re-route demand across surviving facilities, but cannot change prices.  This is Remark 2 (eq. 8).

  When prices are fixed at their nominal value $\hat{p}_i = v_i(1 - \text{fill}_i^{\text{nom}})$, the effective (materialised) demand at node $i$ is fixed at:

  $$\lambda_i = \Lambda_i \cdot D_i(\hat{p}_i) = \Lambda_i \cdot \frac{v_i - \hat{p}_i}{v_i} = \Lambda_i \cdot \text{fill}_i^{\text{nom}}$$

  (eq. 1 and 4).  The optimisation then only re-solves the allocation y subject to reduced capacities.

## Metric

The **Value of Adaptive Pricing (VAP)** quantifies the benefit of pricing flexibility:

$$\text{VAP} = \pi(\text{Scenario a}) - \pi(\text{Scenario b})$$

A higher VAP indicates that pricing flexibility is more valuable — i.e., being forced to hold prices fixed is more costly.

We report VAP under:
1. The **worst-case disruption** for each solution (adversarial analysis)
2. An **average over sampled disruptions** (Monte Carlo analysis)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from rcflp import (
    instancemaker,
    solve_nominal,
    solve_CCG,
    evaluate_second_stage,
    evaluate_fixed_price,
    compute_prices,
    compute_epsilon_scalar,
    worst_case_disruption,
    no_disruption_scenario,
    sample_disruptions,
    compute_risk_metrics,
)

print('Imports OK.')

## 1. Configuration

In [ ]:
In, Jn, Rn   = 10, 8, 2
V_SCALE, W   = 0.75, 10.0
GAMMA, Hn    = 2, 2
N_SAMPLES    = 30
SEED         = 42
TOL, TIME_LIMIT = 0.01, 600
DATA_PATH    = '../dataset.xlsx'

print(f'Instance : I={In}, J={Jn}, R={Rn}, v_scale={V_SCALE}, w={W}')
print(f'Robust   : Gamma={GAMMA}, Hn={Hn}')
print(f'Monte Carlo: N_SAMPLES={N_SAMPLES}, SEED={SEED}')

## 2. Solve nominal and robust problems

We first solve the **nominal problem** (no disruptions) and the **robust problem** (C&CG, eq. 12 of the paper).
The nominal solution serves as a warm-start for C&CG.

In [ ]:
inst  = instancemaker(In, Jn, Rn, V_SCALE, W, data_path=DATA_PATH)
nom   = solve_nominal(inst)
x_nom = nom['x_jr']
print(f'Nominal profit: {nom["profit"]:,.1f}  (runtime: {nom["runtime"]:.1f}s)')

ccg   = solve_CCG(
    inst, GAMMA, Hn,
    x_init=x_nom,
    tol=TOL,
    time_limit=TIME_LIMIT,
    verbose=True,
)
x_rob = ccg['x_jr']
print(f'Robust profit (LB): {ccg["profit_LB"]:,.1f}  '
      f'(converged={ccg["converged"]}, iters={ccg["n_iter"]}, runtime={ccg["runtime"]:.1f}s)')

## 3. Nominal prices (pre-disruption baseline)

To define the fixed-price baseline for Scenario b, we need the **pre-disruption prices** $\hat{p}_i$.
We evaluate each solution under the *no-disruption scenario* (full capacity, ε_j = 1 for all j)
to extract the optimal allocation $y^{\text{nom}}$ and recover prices via eq. (4):

$$\hat{p}_i = v_i \left(1 - \sum_j y_{ij}^{\text{nom}}\right)$$

These prices $\hat{p}_i$ are the ones the provider would charge in the absence of disruption.
In Scenario b they remain locked even after disruption occurs.

In [ ]:
eps0 = no_disruption_scenario(inst, Hn)

# Evaluate nominal x under no disruption to extract optimal allocation and prices
nom_nodis = evaluate_second_stage(inst, x_nom, eps0, Hn)
p_nom     = nom_nodis['prices']     # p̂_i = v_i*(1 - Σ_j y_ij^nom)  via eq. (4)
fill_nom  = nom_nodis['fill_rate']

print('Nominal prices p̂_i (pre-disruption, nominal x):')
print(f'  {"Node":>5}  {"v_i":>8}  {"price":>8}  {"fill":>8}  {"eff_demand":>12}')
for i in sorted(inst['I']):
    eff_d = inst['demand'][i] * fill_nom[i]
    print(f'  {i:>5}  {inst["value_max"][i]:>8.2f}  {p_nom[i]:>8.2f}  '
          f'{fill_nom[i]:>8.3f}  {eff_d:>12.2f}')

## 4. Worst-case disruption analysis

We find the adversarial worst-case disruption ε* for the **nominal solution** x_nom
(maximising recourse cost over the uncertainty set Ξ, Corollary 2 eq. 17).
Then we evaluate both Scenario a and Scenario b under this disruption.

In [ ]:
eps_wc_nom, rc_nom = worst_case_disruption(inst, x_nom, GAMMA, Hn)

eps_j_nom = compute_epsilon_scalar(eps_wc_nom, inst['J'], Hn)
print('Worst-case disruption for nominal solution:')
disrupted = {j: round(1 - eps_j_nom[j], 3) for j in inst['J'] if eps_j_nom[j] < 0.99}
print(f'  Disrupted facilities (capacity reduction fraction): {disrupted}')
print(f'  Recourse cost (excl. fixed): {rc_nom:,.1f}')

### 4a. Scenario a: Full recourse (adaptive pricing)

After the disruption, the provider **jointly re-optimises allocation y and prices p**.
This is the standard second-stage SOCP (Corollary 1, eq. 15).  Prices adjust to
reflect the reduced capacity, so demand is partially shed via price signals.

In [ ]:
res_a_nom = evaluate_second_stage(inst, x_nom, eps_wc_nom, Hn)

print(f'Scenario a (full recourse) — Nominal x — Profit: {res_a_nom["profit"]:,.1f}')
print(f'  Fill rates: {{\'i\': fill}}')
for i in sorted(inst['I']):
    print(f'  Node {i}: price={res_a_nom["prices"][i]:.2f}, '
          f'fill={res_a_nom["fill_rate"][i]:.3f}')

### 4b. Scenario b: Fixed prices (allocation only)

Prices are **locked at the pre-disruption nominal values** $\hat{p}_i$.
The effective demand is therefore fixed at $\lambda_i = \Lambda_i \cdot (v_i - \hat{p}_i)/v_i$
(which equals $\Lambda_i \cdot \text{fill}_i^{\text{nom}}$ when nominal prices are used).

The provider can only **re-route demand** across surviving facilities to minimise
transport and congestion costs — no price adjustment is allowed (Remark 2, eq. 8).

In [ ]:
res_b_nom = evaluate_fixed_price(inst, x_nom, p_nom, eps_wc_nom, Hn)

VAP_nom = res_a_nom['profit'] - res_b_nom['profit']

print(f'Scenario b (fixed price) — Nominal x — Profit: {res_b_nom["profit"]:,.1f}')
print(f'Value of Adaptive Pricing (VAP) for Nominal x:  {VAP_nom:,.1f}')
print()
print('Effective demand (fixed at nominal fill level):')
for i in sorted(inst['I']):
    print(f'  Node {i}: eff_demand={res_b_nom["effective_demand"][i]:.2f}, '
          f'fill={res_b_nom["fill_rate"][i]:.3f}')

## 5. Same analysis for robust solution

We repeat the VAP analysis for the **robust solution** x_rob obtained from C&CG.
The pre-disruption prices for Scenario b are taken from the robust solution evaluated
under no disruption — consistent with the perspective of a robust planner who also
cannot adjust prices after disruption.

In [ ]:
# Find worst-case disruption for robust x
eps_wc_rob, rc_rob = worst_case_disruption(inst, x_rob, GAMMA, Hn)

eps_j_rob = compute_epsilon_scalar(eps_wc_rob, inst['J'], Hn)
disrupted_rob = {j: round(1 - eps_j_rob[j], 3) for j in inst['J'] if eps_j_rob[j] < 0.99}
print(f'Worst-case disruption for robust solution: {disrupted_rob}')
print(f'  Recourse cost (excl. fixed): {rc_rob:,.1f}')
print()

# Nominal prices for robust x: evaluate under no disruption
rob_nodis = evaluate_second_stage(inst, x_rob, eps0, Hn)
p_rob     = rob_nodis['prices']   # pre-disruption prices for robust solution

# Scenario a: full recourse
res_a_rob = evaluate_second_stage(inst, x_rob, eps_wc_rob, Hn)

# Scenario b: prices locked at robust pre-disruption values
res_b_rob = evaluate_fixed_price(inst, x_rob, p_rob, eps_wc_rob, Hn)

VAP_rob = res_a_rob['profit'] - res_b_rob['profit']

print(f'Scenario a (full recourse) — Robust x — Profit: {res_a_rob["profit"]:,.1f}')
print(f'Scenario b (fixed price)   — Robust x — Profit: {res_b_rob["profit"]:,.1f}')
print(f'Value of Adaptive Pricing (VAP) for Robust x:   {VAP_rob:,.1f}')

## 6. Price comparison across scenarios

The table below shows how prices change between the pre-disruption baseline and each
recourse scenario.  In Scenario b, prices are unchanged by definition ($\hat{p}_i^{\text{nom}}$
or $\hat{p}_i^{\text{rob}}$).  In Scenario a, adaptive prices typically *increase* under
disruption (reduced capacity → less allocation → higher price via eq. 4).

In [ ]:
rows = []
for i in sorted(inst['I']):
    rows.append({
        'Node':               i,
        'v_i':                inst['value_max'][i],
        # Pre-disruption prices
        'p_nom (pre-dis)':    round(p_nom[i], 3),
        'p_rob (pre-dis)':    round(p_rob[i], 3),
        # Scenario a: adaptive prices
        'p_ScenA_nom':        round(res_a_nom['prices'][i], 3),
        'p_ScenA_rob':        round(res_a_rob['prices'][i], 3),
        # Scenario b: prices fixed at pre-disruption values
        'p_ScenB_nom':        round(p_nom[i], 3),   # unchanged by construction
        'p_ScenB_rob':        round(p_rob[i], 3),   # unchanged by construction
        # Fill rates
        'fill_nom_nodis':     round(fill_nom[i], 3),
        'fill_ScenA_nom':     round(res_a_nom['fill_rate'][i], 3),
        'fill_ScenB_nom':     round(res_b_nom['fill_rate'][i], 3),
        'fill_ScenA_rob':     round(res_a_rob['fill_rate'][i], 3),
        'fill_ScenB_rob':     round(res_b_rob['fill_rate'][i], 3),
    })

df_prices = pd.DataFrame(rows).set_index('Node')
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', None)
print('=== Price and Fill-rate Comparison ===')
print(df_prices.to_string())

## 7. Cost breakdown comparison

We decompose total profit into its components for each of the four (solution, scenario) combinations:
- **Fixed cost**: facility opening / capacity costs $\sum_{j,r} f_{jr} x_{jr}$ (sunk)
- **Revenue**: $\sum_{i,j} \lambda_i p_i y_{ij}$
- **Transport**: $\sum_{i,j} \lambda_i d_{ij} y_{ij}$
- **Congestion**: $\sum_j w_j c_j$
- **Profit**: Revenue − Transport − Fixed − Congestion

In [ ]:
breakdown_rows = [
    ('NomX_ScenA', res_a_nom['breakdown']),
    ('NomX_ScenB', res_b_nom['breakdown']),
    ('RobX_ScenA', res_a_rob['breakdown']),
    ('RobX_ScenB', res_b_rob['breakdown']),
]

bd_records = []
for label, bd in breakdown_rows:
    bd_records.append({
        'Scenario':   label,
        'Fixed':      bd['fixed_cost'],
        'Revenue':    bd['revenue'],
        'Transport':  bd['transport'],
        'Congestion': bd['congestion'],
        'Profit':     bd['profit'],
    })

df_bd = pd.DataFrame(bd_records).set_index('Scenario')
pd.set_option('display.float_format', '{:,.1f}'.format)
print('=== Cost Breakdown Comparison (Worst-case Disruption) ===')
print(df_bd.to_string())
print()
print('VAP (NomX): ScenA - ScenB =', f"{res_a_nom['profit'] - res_b_nom['profit']:,.1f}")
print('VAP (RobX): ScenA - ScenB =', f"{res_a_rob['profit'] - res_b_rob['profit']:,.1f}")

## 8. Average-case analysis

The worst-case disruption is the adversarial bound.  To assess how adaptive pricing
performs on *average*, we sample $N = 30$ random disruption scenarios from the
uncertainty set $\Xi$ and compute VAP for each.

In [ ]:
scenarios = sample_disruptions(inst, GAMMA, Hn, N_SAMPLES, SEED)

profits_a_nom_list = []
profits_b_nom_list = []
profits_a_rob_list = []
profits_b_rob_list = []
vap_nom_list = []
vap_rob_list = []

for k, eps in enumerate(scenarios):
    ra_n = evaluate_second_stage(inst, x_nom, eps, Hn)
    rb_n = evaluate_fixed_price(inst, x_nom, p_nom, eps, Hn)
    ra_r = evaluate_second_stage(inst, x_rob, eps, Hn)
    rb_r = evaluate_fixed_price(inst, x_rob, p_rob, eps, Hn)
    profits_a_nom_list.append(ra_n['profit'])
    profits_b_nom_list.append(rb_n['profit'])
    profits_a_rob_list.append(ra_r['profit'])
    profits_b_rob_list.append(rb_r['profit'])
    vap_nom_list.append(ra_n['profit'] - rb_n['profit'])
    vap_rob_list.append(ra_r['profit'] - rb_r['profit'])

print(f'Average VAP — Nominal x: {np.mean(vap_nom_list):,.1f}  (std: {np.std(vap_nom_list):,.1f})')
print(f'Average VAP — Robust  x: {np.mean(vap_rob_list):,.1f}  (std: {np.std(vap_rob_list):,.1f})')
print()
print(f'Min VAP — Nominal x: {np.min(vap_nom_list):,.1f}  |  Max: {np.max(vap_nom_list):,.1f}')
print(f'Min VAP — Robust  x: {np.min(vap_rob_list):,.1f}  |  Max: {np.max(vap_rob_list):,.1f}')

## 9. Summary table

Consolidated summary of profits and VAP under both worst-case and average-case disruptions.

In [ ]:
summary_data = {
    'Analysis':   ['Worst-case', f'Average-case (N={N_SAMPLES})'],
    'NomX_ScenA': [res_a_nom['profit'], np.mean(profits_a_nom_list)],
    'NomX_ScenB': [res_b_nom['profit'], np.mean(profits_b_nom_list)],
    'VAP_Nom':    [VAP_nom,             np.mean(vap_nom_list)],
    'RobX_ScenA': [res_a_rob['profit'], np.mean(profits_a_rob_list)],
    'RobX_ScenB': [res_b_rob['profit'], np.mean(profits_b_rob_list)],
    'VAP_Rob':    [VAP_rob,             np.mean(vap_rob_list)],
}

df_summary = pd.DataFrame(summary_data).set_index('Analysis')
pd.set_option('display.float_format', '{:,.1f}'.format)
print('=== Summary: Profits and Value of Adaptive Pricing ===')
print(df_summary.to_string())

## 8.1 Risk Metrics

We compute risk-oriented metrics from the Monte Carlo sample to characterise the *distribution*
of profits and VAP across disruption scenarios — not just the mean.

Three paired comparisons (nominal vs robust):
- **Scenario A** (adaptive pricing): how does risk profile differ between x_nom and x_rob?
- **Scenario B** (fixed-price recourse): same question under price rigidity
- **VAP**: how does the *value of pricing flexibility* vary across scenarios?

In [ ]:
rm_a   = compute_risk_metrics(profits_a_nom_list, profits_a_rob_list)
rm_b   = compute_risk_metrics(profits_b_nom_list, profits_b_rob_list)
rm_vap = compute_risk_metrics(vap_nom_list,       vap_rob_list)

METRIC_LABELS = {
    'mean_profit':  'Mean profit',
    'min_profit':   'Min profit',
    'pct5_profit':  '5th pct profit',
    'cvar5':        'CVaR 5%',
    'cvar10':       'CVaR 10%',
    'prob_loss':    'P(profit<0)',
    'mean_regret':  'Mean regret',
    'max_regret':   'Max regret',
}

def print_risk_table(rm, title):
    print(f'\n{title}  (N={rm["n_scenarios"]} scenarios)')
    print(f'{"Metric":<22}  {"Nominal":>12}  {"Robust":>12}')
    print('-' * 50)
    for key, label in METRIC_LABELS.items():
        vn = rm['nominal'][key]
        vr = rm['robust'][key]
        if key == 'prob_loss':
            print(f'{label:<22}  {vn:>11.1%}  {vr:>11.1%}')
        else:
            print(f'{label:<22}  {vn:>12,.1f}  {vr:>12,.1f}')

print_risk_table(rm_a,   '=== Scenario A — Adaptive Pricing ===')
print_risk_table(rm_b,   '=== Scenario B — Fixed-Price Recourse ===')
print_risk_table(rm_vap, '=== VAP Distribution ===')

## 9. Sensitivity Analysis

We sweep over instance parameters to study how VAP and risk metrics change with:
- **v_scale** (willingness-to-pay multiplier)
- **w** (congestion cost)
- **Γ** (uncertainty budget)

For each combination we solve the nominal and robust problems, evaluate over
`N_SAMPLES_SENS` sampled disruptions, and record profit/VAP statistics.

In [ ]:
V_SCALE_LIST   = [0.50, 0.75, 1.00]
W_LIST         = [5.0,  10.0, 20.0]
GAMMA_LIST     = [1,    2,    3]
N_SAMPLES_SENS = 30
SEED_SENS      = 42

n_combos = len(V_SCALE_LIST) * len(W_LIST) * len(GAMMA_LIST)
print(f'Sensitivity grid: {len(V_SCALE_LIST)} v_scales x {len(W_LIST)} w values x {len(GAMMA_LIST)} Gamma values')
print(f'= {n_combos} combinations x {N_SAMPLES_SENS} OOS scenarios each')

In [ ]:
import itertools
import time as _time

sens_rows = []
combo_list = list(itertools.product(V_SCALE_LIST, W_LIST, GAMMA_LIST))
total = len(combo_list)

for run_idx, (v, w, gam) in enumerate(combo_list, 1):
    t_run = _time.time()
    print(f'[{run_idx}/{total}]  v={v}, w={w}, Gamma={gam}', end='  ', flush=True)

    inst_s  = instancemaker(In, Jn, Rn, v, w, data_path=DATA_PATH)
    nom_s   = solve_nominal(inst_s)
    x_nom_s = nom_s['x_jr']

    ccg_s   = solve_CCG(inst_s, gam, Hn, x_init=x_nom_s, tol=TOL, time_limit=TIME_LIMIT)
    x_rob_s = ccg_s['x_jr']

    eps0_s   = no_disruption_scenario(inst_s, Hn)
    nom_nd_s = evaluate_second_stage(inst_s, x_nom_s, eps0_s, Hn)
    rob_nd_s = evaluate_second_stage(inst_s, x_rob_s, eps0_s, Hn)
    p_nom_s  = nom_nd_s['prices']
    p_rob_s  = rob_nd_s['prices']

    eps_wc_nom_s, _ = worst_case_disruption(inst_s, x_nom_s, gam, Hn)
    eps_wc_rob_s, _ = worst_case_disruption(inst_s, x_rob_s, gam, Hn)

    wc_a_nom_s = evaluate_second_stage(inst_s, x_nom_s, eps_wc_nom_s, Hn)
    wc_b_nom_s = evaluate_fixed_price(inst_s, x_nom_s, p_nom_s, eps_wc_nom_s, Hn)
    wc_a_rob_s = evaluate_second_stage(inst_s, x_rob_s, eps_wc_rob_s, Hn)
    wc_b_rob_s = evaluate_fixed_price(inst_s, x_rob_s, p_rob_s, eps_wc_rob_s, Hn)

    wc_vap_nom_s = wc_a_nom_s['profit'] - wc_b_nom_s['profit']
    wc_vap_rob_s = wc_a_rob_s['profit'] - wc_b_rob_s['profit']

    scen_s = sample_disruptions(inst_s, gam, Hn, N_SAMPLES_SENS, SEED_SENS)

    pa_nom_s, pb_nom_s, pa_rob_s, pb_rob_s = [], [], [], []
    for eps_s in scen_s:
        pa_nom_s.append(evaluate_second_stage(inst_s, x_nom_s, eps_s, Hn)['profit'])
        pb_nom_s.append(evaluate_fixed_price(inst_s, x_nom_s, p_nom_s, eps_s, Hn)['profit'])
        pa_rob_s.append(evaluate_second_stage(inst_s, x_rob_s, eps_s, Hn)['profit'])
        pb_rob_s.append(evaluate_fixed_price(inst_s, x_rob_s, p_rob_s, eps_s, Hn)['profit'])

    vap_nom_s = [a - b for a, b in zip(pa_nom_s, pb_nom_s)]
    vap_rob_s = [a - b for a, b in zip(pa_rob_s, pb_rob_s)]

    rm_a_s   = compute_risk_metrics(pa_nom_s, pa_rob_s)
    rm_b_s   = compute_risk_metrics(pb_nom_s, pb_rob_s)
    rm_vap_s = compute_risk_metrics(vap_nom_s, vap_rob_s)

    elapsed = _time.time() - t_run
    print(f'done ({elapsed:.0f}s)')

    row = {
        'v_scale': v, 'w': w, 'gamma': gam,
        'nom_profit':    nom_s['profit'],
        'rob_profit_LB': ccg_s['profit_LB'],
        'wc_vap_nom':    wc_vap_nom_s,
        'wc_vap_rob':    wc_vap_rob_s,
        'avg_vap_nom':   float(np.mean(vap_nom_s)),
        'avg_vap_rob':   float(np.mean(vap_rob_s)),
        'min_vap_nom':   float(np.min(vap_nom_s)),
        'min_vap_rob':   float(np.min(vap_rob_s)),
        'pct5_vap_nom':  float(np.percentile(vap_nom_s, 5)),
        'pct5_vap_rob':  float(np.percentile(vap_rob_s, 5)),
    }

    for prefix, rm, side in [
        ('a_nom',   rm_a_s,   'nominal'),
        ('a_rob',   rm_a_s,   'robust'),
        ('b_nom',   rm_b_s,   'nominal'),
        ('b_rob',   rm_b_s,   'robust'),
        ('vap_nom', rm_vap_s, 'nominal'),
        ('vap_rob', rm_vap_s, 'robust'),
    ]:
        for m in ['mean_profit', 'min_profit', 'pct5_profit', 'cvar5', 'cvar10',
                  'prob_loss', 'mean_regret', 'max_regret']:
            row[f'{prefix}_{m}'] = rm[side][m]

    sens_rows.append(row)

df_sens = pd.DataFrame(sens_rows)
print(f'\nSensitivity analysis complete. Shape: {df_sens.shape}')
preview_cols = ['v_scale', 'w', 'gamma', 'wc_vap_nom', 'wc_vap_rob', 'avg_vap_nom', 'avg_vap_rob']
print(df_sens[preview_cols].to_string())

## 10. Save Sensitivity Results to Excel

In [ ]:
EXCEL_PATH = '03_adaptive_pricing_sensitivity.xlsx'

with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
    df_sens.to_excel(writer, sheet_name='Full_Results', index=False)

    pivot_specs = [
        ('wc_vap_nom',    'WC_VAP_Nom'),
        ('wc_vap_rob',    'WC_VAP_Rob'),
        ('avg_vap_nom',   'Avg_VAP_Nom'),
        ('avg_vap_rob',   'Avg_VAP_Rob'),
        ('a_nom_cvar5',   'ScenA_Nom_CVaR5'),
        ('a_rob_cvar5',   'ScenA_Rob_CVaR5'),
        ('b_nom_cvar5',   'ScenB_Nom_CVaR5'),
        ('b_rob_cvar5',   'ScenB_Rob_CVaR5'),
        ('vap_nom_cvar5', 'VAP_Nom_CVaR5'),
        ('vap_rob_cvar5', 'VAP_Rob_CVaR5'),
        ('a_nom_prob_loss',  'ScenA_Nom_ProbLoss'),
        ('a_rob_prob_loss',  'ScenA_Rob_ProbLoss'),
    ]
    for col, sheet in pivot_specs:
        piv = df_sens.pivot_table(index='w', columns=['v_scale', 'gamma'], values=col)
        piv.to_excel(writer, sheet_name=sheet)

print(f'Saved to {EXCEL_PATH}')
print(df_sens[['v_scale','w','gamma','avg_vap_nom','avg_vap_rob','a_nom_cvar5','a_rob_cvar5']].to_string())

## 11. Sensitivity Plots — Value of Adaptive Pricing

## 10. Plots

### Plot 1: Worst-case profits — Scenario a vs b for nominal and robust solutions
### Plot 2: Distribution of VAP over sampled disruptions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Grouped bar chart — worst-case profits ─────────────────────────
ax1 = axes[0]

groups     = ['Nominal x', 'Robust x']
scen_a     = [res_a_nom['profit'], res_a_rob['profit']]
scen_b     = [res_b_nom['profit'], res_b_rob['profit']]

x_pos  = np.arange(len(groups))
width  = 0.35

bars_a = ax1.bar(x_pos - width/2, scen_a, width, label='Scenario a\n(Adaptive pricing)',
                 color='steelblue', alpha=0.85, edgecolor='k', linewidth=0.7)
bars_b = ax1.bar(x_pos + width/2, scen_b, width, label='Scenario b\n(Fixed prices)',
                 color='coral', alpha=0.85, edgecolor='k', linewidth=0.7)

# Annotate with VAP arrows
for idx, (sa, sb) in enumerate(zip(scen_a, scen_b)):
    vap = sa - sb
    mid_y = max(sa, sb) + abs(sa) * 0.02
    ax1.annotate(
        f'VAP={vap:,.0f}',
        xy=(x_pos[idx], mid_y),
        ha='center', va='bottom', fontsize=9, color='darkgreen', fontweight='bold',
    )

ax1.set_xticks(x_pos)
ax1.set_xticklabels(groups, fontsize=11)
ax1.set_ylabel('Profit', fontsize=11)
ax1.set_title('Worst-case Profit: Scenario a vs b', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax1.grid(axis='y', linestyle='--', alpha=0.5)


# ── Plot 2: Histogram — distribution of VAP ────────────────────────────────
ax2 = axes[1]

bins = np.linspace(
    min(min(vap_nom_list), min(vap_rob_list)) - 1,
    max(max(vap_nom_list), max(vap_rob_list)) + 1,
    15,
)

ax2.hist(vap_nom_list, bins=bins, alpha=0.6, color='steelblue',
         label=f'Nominal x  (mean={np.mean(vap_nom_list):,.0f})', edgecolor='k', linewidth=0.6)
ax2.hist(vap_rob_list, bins=bins, alpha=0.6, color='coral',
         label=f'Robust x   (mean={np.mean(vap_rob_list):,.0f})', edgecolor='k', linewidth=0.6)

ax2.axvline(np.mean(vap_nom_list), color='steelblue', linestyle='--', linewidth=1.5)
ax2.axvline(np.mean(vap_rob_list), color='coral',     linestyle='--', linewidth=1.5)
ax2.axvline(0, color='black', linestyle='-', linewidth=1.0, alpha=0.5)

ax2.set_xlabel('VAP = Profit(ScenA) - Profit(ScenB)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title(f'Distribution of VAP over {N_SAMPLES} sampled disruptions', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', linestyle='--', alpha=0.5)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('03_adaptive_pricing_wc_plots.pdf', bbox_inches='tight')
plt.savefig('03_adaptive_pricing_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to 03_adaptive_pricing_plots.png')

## Interpretation

### Key findings

1. **VAP quantifies the cost of price rigidity.**  The difference between Scenario a
   and Scenario b profit (VAP) captures precisely the loss incurred when the provider
   cannot adapt prices after a disruption.  A positive VAP confirms that pricing
   flexibility is genuinely valuable — locking prices at pre-disruption levels leads
   to strictly lower profit.

2. **Why is Scenario b worse?**  Under fixed prices, the effective demand $\lambda_i$
   is determined by the pre-disruption price signal and does not respond to the
   reduced capacity.  The provider may face demand that exceeds surviving capacity,
   forcing more congestion or demand shedding *without* the compensating revenue
   uplift that higher adaptive prices would generate.  In Scenario a, the provider
   can raise prices to reduce demand *and* improve revenue simultaneously.

3. **Robust vs nominal solution.**  The robust solution (x_rob) is designed to hedge
   against worst-case disruptions.  Under its own worst-case scenario, VAP_rob
   reflects how much pricing flexibility is worth in the worst case relevant to a
   robust operator.  Because the robust solution opens facilities in a more
   disruption-resilient configuration, the gap between the two scenarios may differ
   from the nominal case.

4. **Average-case analysis.**  The sampled VAP distribution shows how pricing
   flexibility performs across a realistic range of disruptions, not just the
   adversarial extreme.  A consistently positive VAP across samples confirms the
   robustness of the finding.

5. **Policy implication.**  Section 3.3 of the paper argues that adaptive pricing is
   a valuable operational lever alongside facility location.  The VAP metric provides
   a concrete monetary estimate of the benefit of building price-flexibility into the
   recourse policy, which informs the business case for dynamic pricing mechanisms.